In [2]:
import pandas as pd
from transformers import CLIPTokenizer, CLIPTextModel
import numpy as np
import pickle
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio, random
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity

"Pierre-Auguste Renoir"
"Käthe Kollwitz"
"Max Ernst"
"Karel Appel"

In [3]:
Artist_name ="Max Ernst"

In [4]:
claude_label = pd.read_excel(f"LLM Request\\comment_annotation_claude_{Artist_name.split(" ")[-1]}.xlsx")
gemini_label = pd.read_excel(f"LLM Request\\comment_annotation_gemini_{Artist_name.split(" ")[-1]}.xlsx")
openai_label = pd.read_excel(f"LLM Request\\comment_annotation_openai_{Artist_name.split(" ")[-1]}.xlsx")

In [5]:
gemini_label.shape

(1000, 10)

In [6]:
full_df = claude_label.merge(gemini_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=("_claude","_gemini"))

In [7]:
full_df

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,artistic_value_answer_gemini,artistic_value_comment_gemini,creativity_answer_gemini,creativity_comment_gemini
0,287,La Forêt,Max,Ernst,1925,German,High,Max Ernst was one of the most materially inven...,Yes,Ernst invented the frottage technique at Porni...,High,"Max Ernst's ""La Forêt"" (1925) is widely regard...",Yes,"Max Ernst's ""La Forêt"" (1925) is exceptionally..."
1,875,Tremblement de terre printanier or Trois tremb...,Max,Ernst,1964,German,High,This exceptional painting from Ernst's mature ...,Yes,The work invites interpretation as expressing ...,High,"Max Ernst's ""Tremblement de terre printanier"" ...",Yes,"""Tremblement de terre printanier"" demonstrates..."
2,2196,Dormeuse,Max,Ernst,1955,German,"authoritative commentary on Max Ernst's ""Dorme...",What I cannot provide:**\nWithout specific sch...,"for authoritative commentary on Max Ernst's ""D...",What I cannot provide:**\nWithout specific sch...,High,"Max Ernst's ""Dormeuse"" (1955) is widely regard...",Yes,"""Dormeuse"" demonstrates significant creativity..."
3,3062,Le chant de la grenouille,Max,Ernst,1957,German,authoritative commentary on this Max Ernst art...,I cannot provide the analysis you requested** ...,for authoritative commentary on this Max Ernst...,I cannot provide the analysis you requested** ...,High,"Max Ernst's ""Le chant de la grenouille"" (1957)...",Yes,"""Le chant de la grenouille"" demonstrates signi..."
4,3491,Drapeau,Max,Ernst,1967,German,"scholarly commentary on Max Ernst's ""Drapeau"" ...",Without authoritative source material on this ...,"for scholarly commentary on Max Ernst's ""Drape...",Without authoritative source material on this ...,High,"Max Ernst's ""Drapeau"" (1967), while not as ext...",Yes,"Max Ernst's ""Drapeau"" (1967) is a product of a..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,428142056,Elfo III,Max,Ernst,1966,German,Moderate to High,"""Elfo III"" is a glass sculpture created in 196...",Conditional,The available sources do not provide sufficien...,High,"Max Ernst's ""Elfo III,"" created in 1966, is a ...",Yes,"""Elfo III"" is a creative work due to its novel..."
996,428143973,La ballade du soldat (bk by Georges Ribemont D...,Max,Ernst,1972,German,High,This 1972 illustrated book with lithographs fo...,Limited,While Ernst's contributions to artistic innova...,High,"Max Ernst's ""La ballade du soldat,"" created in...",Yes,"""La ballade du soldat"" by Max Ernst demonstrat..."
997,428154983,Au liège rendu par la mer,Max,Ernst,1969,German,scholarly commentary on this Max Ernst artwork...,Without access to authoritative critical sourc...,for scholarly commentary on this Max Ernst art...,Without access to authoritative critical sourc...,High,"Max Ernst's ""Au liège rendu par la mer"" (1969)...",Yes,"Max Ernst's ""Au liège rendu par la mer"" demons..."
998,428154984,Endlose Spiele bereiten sich vor,Max,Ernst,1972,German,NaN,NaN,NaN,NaN,High,"Max Ernst's 1972 silkscreen ""Endlose Spiele be...",Yes,Max Ernst's entire artistic career is marked b...


In [8]:
full_df = full_df.merge(openai_label.drop(columns=["title",'first','last','workyear from',"nationality"]),on=["artwork id"],how="outer",suffixes=(None,"_openai"))

In [9]:
claude_embed = np.load(f"clip_embeddings_claude_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)
gemini_embed = np.load(f"clip_embeddings_gemini_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)
openai_embed = np.load(f"clip_embeddings_openai_{Artist_name.split(" ")[-1]}.npy",allow_pickle=True)

# Golden Set

Golden set is a set of samples that have the same answers for "type" and "creative". To add to confidence, only samples with cosine similarity above a given threshold are kept.

In [24]:
threshold = 0.65

In [25]:
consistent_creative=[]
consistent_artist=[]
consistent_overall=[]
for i in range(full_df.shape[0]):
    row = full_df.iloc[i]
    if (row.artistic_value_answer_claude == row.artistic_value_answer.strip()) & (row.artistic_value_answer.strip() == row.artistic_value_answer_gemini):
        artist_con=1
        consistent_artist.append(1)
    else:
        artist_con=0
        consistent_artist.append(0)
    if (row.creativity_answer_claude == row.creativity_answer.strip()) & (row.creativity_answer.strip() == row.creativity_answer_gemini):
        creative=1
        consistent_creative.append(1)
    else:
        creative=0
        consistent_creative.append(0)

    if artist_con+creative==2:
        consistent_overall.append(1)
    else:
        consistent_overall.append(0)

In [26]:
np.sum(consistent_artist)

np.int64(341)

In [27]:
np.sum(consistent_creative)

np.int64(307)

In [28]:
np.sum(consistent_overall)

np.int64(301)

In [29]:
checking=full_df.copy()
checking["creative_consist"]=consistent_creative
checking["artistic_consist"]=consistent_artist
checking["overall_consist"]=consistent_overall

In [30]:
print(f"""
For type, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
""")

print(f"""
For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
""")


For type, there are 341 samples that are consistent.


For creative, there are 307 samples that are consistent.



In [31]:
print(f"""
When looking at only type and creative, there are {np.sum(checking["overall_consist"])} samples that are consistent.
""")


When looking at only type and creative, there are 301 samples that are consistent.



In [32]:
embed_consistent_creative=[]
embed_consistent_artistic=[]
embed_consistent_overall=[]
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(openai_embed.shape[0]):
    artistic_sim= cosine_similarity([claude_embed[i][0],gemini_embed[i][0],openai_embed[i][0]])
    if (artistic_sim > threshold).all():
        artistic_con=1
        embed_consistent_artistic.append(1)
    else:
        artistic_con=0
        embed_consistent_artistic.append(0)
    creative_sim= cosine_similarity([claude_embed[i][1],gemini_embed[i][1],openai_embed[i][1]])
    if (creative_sim > threshold).all():
        creative=1
        embed_consistent_creative.append(1)
    else:
        creative=0
        embed_consistent_creative.append(0)

    
    if artistic_con+creative==2:
        embed_consistent_overall.append(1)
    else:
        embed_consistent_overall.append(0)
    if i%2500==0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Currently at {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: End")

2025-12-16 05:02:34: Start
2025-12-16 05:02:34: Currently at 0
2025-12-16 05:02:34: End


In [33]:
checking["embed_artistic_consist"]=embed_consistent_artistic
checking["embed_creative_consist"]=embed_consistent_creative
checking["embed_overall_consist"]=embed_consistent_overall

In [34]:
print(f"""
[Easy] For artistic, there are {np.sum(checking["artistic_consist"])} samples that are consistent.
[Hard] For artistic, there are {checking[(checking["artistic_consist"]==1) & (checking["embed_artistic_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy] For creative, there are {np.sum(checking["creative_consist"])} samples that are consistent.
[Hard] For creative, there are {checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)].shape[0]} samples that are consistent.
""")

print(f"""
[Easy]For overall, there are {np.sum(checking["overall_consist"])} samples that are consistent.
[Hard] For overall, there are {checking[(checking["overall_consist"]==1) & (checking["embed_overall_consist"]==1)].shape[0]} samples that are consistent.
""")


[Easy] For artistic, there are 341 samples that are consistent.
[Hard] For artistic, there are 183 samples that are consistent.


[Easy] For creative, there are 307 samples that are consistent.
[Hard] For creative, there are 17 samples that are consistent.


[Easy]For overall, there are 301 samples that are consistent.
[Hard] For overall, there are 8 samples that are consistent.



In [35]:
golden_set_move_creative = checking[(checking["creative_consist"]==1) & (checking["embed_creative_consist"]==1)]

In [36]:
golden_set_move_creative.shape

(17, 24)

In [37]:
golden_set_move_creative.to_excel(f"golden_set_move_creative_{int(threshold*100)}_{Artist_name.split(" ")[-1]}.xlsx",index=False)

# Double Check

In [22]:
golden_set_move_creative = pd.read_excel("golden_set_move_creative_65.xlsx")

In [23]:
golden_set_move_creative

,artwork id,title,first,last,workyear from,nationality,artistic_value_answer_claude,artistic_value_comment_claude,creativity_answer_claude,creativity_comment_claude,...,artistic_value_answer,artistic_value_comment,creativity_answer,creativity_comment,creative_consist,artistic_consist,overall_consist,embed_artistic_consist,embed_creative_consist,embed_overall_consist
0,1596,Le peintre et son modèle,Pablo,Picasso,1964,Spanish,High,"In Picasso's later years, the theme of painter...",Yes,The work demonstrates residual Cubism through ...,...,High,"""Le Peintre et Son Modèle"" is a significant wo...",Yes,"In ""Le Peintre et Son Modèle,"" Picasso innovat...",1,1,1,1,1,1
1,1781,Verre et citron,Pablo,Picasso,1944,Spanish,High,"According to art historian John Richardson, ""t...",Yes,The work belongs to a series of small-scale wo...,...,High,"""Verre et citron"" is a notable example of Pica...",Yes,"""Verre et citron"" exemplifies Picasso's innova...",1,1,1,1,1,1
2,2849,HOMME ASSIS,Pablo,Picasso,1969,Spanish,High,"""Homme Assis"" was painted during Picasso's mos...",Yes,Picasso's objective to paint 'nature' contrast...,...,High,"Pablo Picasso's ""Homme Assis"" (1969) is a sign...",Yes,"""Homme Assis"" exemplifies Picasso's innovative...",1,1,1,1,1,1
3,3100,"Femme assise dans un fauteuil tressé, en gris ...",Pablo,Picasso,1953,Spanish,High,This portrait of Françoise Gilot was painted i...,Yes,The 1953 portrait innovates beyond Picasso's e...,...,High,"""Femme assise dans un fauteuil tressé, en gris...",Yes,"Picasso's ""Femme assise dans un fauteuil tress...",1,1,1,0,1,0
4,3483,DEUX HIRONDELLES,Pablo,Picasso,1932,Spanish,High,"Painted on May 14, 1932 at the height of his c...",Yes,The work demonstrates creativity through surpr...,...,High,"Pablo Picasso's 1932 painting ""Deux Hirondelle...",Yes,"""Deux Hirondelles"" exemplifies Picasso's creat...",1,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1741,424922090,Visage de femme,Pablo,Picasso,1953,Spanish,High,"This ceramic work, likely depicting Jacqueline...",Yes,"Picasso was inspired by those around him, with...",...,High,"""Visage de femme"" (1953) is a glazed ceramic p...",Yes,"""Visage de femme"" demonstrates Picasso's creat...",1,1,1,1,1,1
1742,424922091,Visage d'homme,Pablo,Picasso,1953,Spanish,High,While specific scholarly critique of this 1953...,Yes,Picasso's use of the cast shadow as a pictoria...,...,High,"""Visage d'homme"" (1953) exemplifies Picasso's ...",Yes,"""Visage d'homme"" showcases Picasso's continuou...",1,1,1,1,1,1
1743,424924633,Vase aztèque aux quatre visages,Pablo,Picasso,1957,Spanish,High,Vase Aztèque aux quatre visages captures the s...,Yes,This work captures Picasso's restless need to ...,...,High,"Pablo Picasso's ""Vase Aztèque aux Quatre Visag...",Yes,"Picasso's ""Vase Aztèque aux Quatre Visages"" de...",1,1,1,1,1,1
1744,424927848,Mousquetaire,Pablo,Picasso,1969,Spanish,High,"Between 1966 and 1972, Picasso displayed porte...",Yes,"For Picasso, the musketeer signified the golde...",...,High,"Pablo Picasso's 1969 painting ""Mousquetaire"" e...",Yes,"In ""Mousquetaire,"" Picasso showcases significa...",1,1,1,1,1,1
